# Поглавље 7: Прављење ћаскаћих апликација
## Брзи почетак са Azure OpenAI API-јем


## Преглед
Овај бележник је прилагођен из [Azure OpenAI примера из репозиторијума](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) који укључује бележнике који такође приступају [OpenAI](notebook-openai.ipynb) сервису.

Python OpenAI API ради и са Azure OpenAI, уз неколико измена. Сазнајте више о разликама овде: [Како пребацити између OpenAI и Azure OpenAI крајњих тачака са Python-ом](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

За више примера за брз почетак, погледајте званичну [Azure OpenAI документацију за брзи почетак](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst)


## Садржај  

[Преглед](#overview)  
[Почетак рада са Azure OpenAI Service](#getting-started-with-azure-openai-service)  
[Направите свој први упит](#build-your-first-prompt)  

[Примери употребе](#use-cases)    
[1. Сажмите текст](#summarize-text)  
[2. Класификујте текст](#classify-text)  
[3. Генеришите нове називе производа](#generate-new-product-names)  
[4. Фино подешавање класификатора](#fine-tune-a-classifier)  
[5. Уметаци (Embeddings)](#embeddings)

[Референце](#references)


### Почетак рада са Azure OpenAI услугом

Нови корисници ће морати да [захтевају приступ](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst) Azure OpenAI услузи.  
Након што одобрење буде завршено, корисници могу да се пријаве на Azure портал, креирају ресурс Azure OpenAI услуге и почну да експериментишу са моделима преко студија  

[Одличан ресурс за брзи почетак](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### Направите свој први упит  
Ова кратка вежба пружиће основни увод у слање упита OpenAI моделу за једноставан задатак „сажимање“.


**Кораци**:  
1. Инсталирајте OpenAI библиотеку у вашем Python окружењу  
2. Учитајте стандардне помоћне библиотеке и подесите уобичајене OpenAI безбедносне креденцијале за OpenAI услугу коју сте креирали  
3. Изаберите модел за ваш задатак  
4. Креирајте једноставан упит за модел  
5. Пошаљите ваш захтев моделу преко API-ја!


### 1. Инсталирајте OpenAI


  > [!NOTE] Овај корак није неопходан ако покренете овај нотебук на Codespaces или унутар Devcontainer-а


In [ ]:
%pip install openai python-dotenv

### 2. Увези помоћне библиотеке и направи акредитиве


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. Проналажење одговарајућег модела  
Модели као што су GPT-4o и GPT-4o мини могу да разумеју и генеришу природни језик. Azure OpenAI у Microsoft Foundry нуди низ могућности модела, сваки са различитим нивоима снаге и брзине погодним за различите задатке. 

[Azure OpenAI у Microsoft Foundry моделима](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. Дизајн упита  

"Магија великих језичких модела је у томе што, тренирајући се да минимализују ову грешку предвиђања преко огромних количина текста, модели на крају уче појмове корисне за ова предвиђања. На пример, уче појмове као што су"(1):

* како се пише  
* како функционише граматика  
* како да се препричава  
* како да се одговара на питања  
* како да се води разговор  
* како да се пише на много језика  
* како да се програмира  
* итд.  

#### Како контролисати велики језички модел  
"Од свих уноса великом језичком моделу, далеко највише утиче текстуални упит(1).

Велики језички модели се могу подстакнути да производе излаз на неколико начина:

- Инструкција: Реците моделу шта желите
- Допуна: Натерајте модел да допуни почетак онога што желите
- Демонстрација: Показујте моделу шта желите, са:
- Неколико примера у упиту
- Стотине или хиљаде примера у скупу за фино подешавање



#### Постоје три основне смернице за креирање упита:

**Покажи и реци**. Јасно покажите шта желите било кроз инструкције, примере, или комбинацију оба. Ако желите да модел рангира списак ставки по абецеди или класификује пасус по сентименту, покажите му да је то оно што желите.

**Пружи квалитетне податке**. Ако покушавате да направите класификатор или да добијете модел да следи образац, уверите се да има довољно примера. Обавезно исправите своје примере — модел је углавном довољно паметан да примети основне правописне грешке и да вам одговори, али такође може претпоставити да је то намера и то може утицати на одговор.

**Провери своја подешавања.** Температуре и топ_p поставке контролишу колико је модел детерминистичан у генерисању одговора. Ако тражите одговор где постоји само један тачан одговор, онда је боље поставити их ниже. Ако тражите разноврсније одговоре, онда их можете поставити више. Најчешћа грешка људи са овим подешавањима је што претпостављају да су то контроле „паметности“ или „креативности“.


Извор: https://learn.microsoft.com/azure/ai-services/openai/overview


слика креира ваш први текстуални упит!


### 5. Пошаљи!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Поновите исти позив, како се резултати упоређују?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Резимирање текста  
#### Изазов  
Резимирајте текст додавањем 'tl;dr:' на крај текста. Примећујете како модел разуме како да изврши низ задатака без додатних упутстава. Можете експериментисати са описнијим упутствима од tl;dr како бисте изменили понашање модела и прилагодили добијени резиме(3).  

Недавни рад је показао значајан напредак на многим NLP задацима и референтним тестовима претходним тренирањем на великом корпусу текста, а затим детаљном прилагођавањем на специфичан задатак. Иако је обично архитектонски независан од задатка, ова метода и даље захтева скупове података за детаљно прилагођавање са хиљадама или десетинама хиљада примера. Насупрот томе, људи обично могу да изврше нови језички задатак само на основу неколико примера или једноставних упутстава — нешто са чиме тренутни NLP системи још увек углавном имају проблема. Овде показујемо да повећање размера језичких модела знатно побољшава перформансе у задатцима са неколико примера, понекад чак достижући конкурентност у односу на претходне најбоље методе детаљног прилагођавања. 



Резиме  


# Вежбе за више случајева коришћења  
1. Сажми текст  
2. Класификуј текст  
3. Генериши нове називе производа
4. Ембеддинзи
5. Фино подеси класификатор


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Класификуј текст  
#### Изазов  
Класификуј ставке у категорије дате у време закључивања. У следећем примеру обезбеђујемо и категорије и текст за класификацију у упиту(*playground_reference). 

Питање корисника: Здраво, једно од тастера на мојој лаптоп тастатури недавно се покварило и биће ми потребна замена:

Класификована категорија:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Генеришите нове назive производа
#### Изазов
Креирајте називе производа из пример речи. Овде укључујемо у упиту информације о производу за који ћемо генерисати називе. Такође пружамо сличан пример да покажемо образац који желимо да добијемо. Поставили смо и високу вредност температуре да повећамо насумичност и добијемо иновативније одговоре.

Опис производа: Кућни прављач шејкова
Основне речи: брз, здрав, компактн.
Називи производа: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис производа: Пар обуће који може да стане на било коју величину стопала.
Основне речи: прилагодљив, подходан, свестран.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## Уграђени вектори
Овај одељак ће показати како добити уграђене векторе и пронаћи сличности између речи, реченица и докумената. Да бисте покренули наредне нотебуке, потребно је да развијете модел који користи `text-embedding-ada-002` као основни модел и поставите његово име развоја унутар .env фајла, користећи променљиву `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT`.


### Таксономија модела - Избор модела угњетавања

**Таксономија модела**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (породица угњетавања)  
{capability} --> ada             (сви остали модели угњетавања биће повучени 2024)  
{input-type} --> n/a             (наведено само за моделе претраге)  
{identifier} --> 002             (верзија 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] Следећи корак није неопходан ако покрећете овај ноутбук у Codespaces или унутар Devcontainer-а


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## Поређење чланка из скупa података cnn daily news
извор: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# Референце  
- [Microsoft Foundry - Azure OpenAI модели](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry портал](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# За додатну помоћ  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# Присутни
* Луис Ли  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
